In [ ]:
import pandas as pd
import numpy as np
from statsmodels.regression.mixed_linear_model import MixedLM
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
survey_long = pd.read_csv('stats_out/survey_long.csv')

print(f"Loaded {len(survey_long)} ratings from survey_long.csv")
print(f"\nData structure:")
print(f"Columns: {list(survey_long.columns)}")
print(f"Unique participants: {survey_long['participant_id'].nunique()}")
print(f"Unique models: {sorted(survey_long['model'].unique())}")
print(f"Unique metrics: {sorted(survey_long['metric'].unique())}")

survey_long['explanation_id'] = (survey_long['model'].astype(str) + ':' + survey_long['metric'].astype(str))

print(f"\nUnique model×metric combinations (explanations): {survey_long['explanation_id'].nunique()}")
print(f"Expected: {len(survey_long['model'].unique())} models × {len(survey_long['metric'].unique())} metrics = {len(survey_long['model'].unique()) * len(survey_long['metric'].unique())}")

analysis_df = survey_long.copy()

print(f"Participants (users): {analysis_df['participant_id'].nunique()}")
print(f"Explanations (model×metric): {analysis_df['explanation_id'].nunique()}")
print(f"Total observations: {len(analysis_df)}")
print(f"Mean rating: {analysis_df['score'].mean():.3f}")
print(f"Rating SD: {analysis_df['score'].std():.3f}")

In [ ]:
vc_formula = {"explanation": "0 + C(explanation_id)"}

model = MixedLM.from_formula(
    "score ~ 1",
    data=analysis_df,
    groups=analysis_df["participant_id"],  
    re_formula="1",                        
    vc_formula=vc_formula                   
)

result = model.fit(reml=True)

print(result.summary())

print(f"result.cov_re shape: {result.cov_re.shape if hasattr(result, 'cov_re') and result.cov_re is not None else 'N/A'}")
if hasattr(result, 'cov_re') and result.cov_re is not None and result.cov_re.size > 0:
    print(f"result.cov_re:\n{result.cov_re}")
else:
    print("result.cov_re: Empty or None")
print(f"\nresult.vcomp type: {type(result.vcomp)}")
print(f"result.vcomp: {result.vcomp}")
print(f"\nresult.scale: {result.scale}")

try:
    if hasattr(result, 'cov_re') and result.cov_re is not None:
        if result.cov_re.size > 0:
            sigma2_user = float(result.cov_re.iloc[0, 0])
        else:
            print("cov_re is empty")
            if isinstance(result.vcomp, dict) and 'user' in result.vcomp:
                sigma2_user = float(result.vcomp['user'])
            else:
                raise ValueError("cov_re is empty and user not found in vcomp")
    else:
        raise ValueError("cov_re attribute not found")
except (IndexError, ValueError, AttributeError) as e:
    print(f"\nCould not extract user variance from cov_re: {e}")
    model_user_only = MixedLM.from_formula(
        "score ~ 1",
        data=analysis_df,
        groups=analysis_df["participant_id"],
        re_formula="1"
    )
    result_user_only = model_user_only.fit(reml=True)
    sigma2_user = float(result_user_only.cov_re.iloc[0, 0])
    print(f"Extracted user variance from user-only model: {sigma2_user:.4f}")

try:
    if isinstance(result.vcomp, dict):
        if 'explanation' in result.vcomp:
            sigma2_explanation = float(result.vcomp['explanation'])
        else:
            keys = list(result.vcomp.keys())
            if len(keys) > 0:
                first_key = keys[0]
                sigma2_explanation = float(result.vcomp[first_key])
                print(f"Using variance component key '{first_key}' instead of 'explanation'")
            else:
                raise ValueError("vcomp dictionary is empty")
    elif hasattr(result.vcomp, '__len__') and len(result.vcomp) > 0:
        sigma2_explanation = float(result.vcomp[0])
    else:
        raise ValueError(f"vcomp is not in expected format: {type(result.vcomp)}, value: {result.vcomp}")
except (KeyError, IndexError, ValueError, AttributeError, TypeError) as e:
    print(f"\nCould not extract explanation variance from vcomp: {e}")
    raise

sigma2_residual = float(result.scale)              

print("\nVariance Components")
print(f"\nσ²_user: {sigma2_user:.4f}")
print(f"\nσ²_explanation (model×metric): {sigma2_explanation:.4f}")
print(f"\nσ²_residual: {sigma2_residual:.4f}")

total_variance = sigma2_user + sigma2_explanation + sigma2_residual
print(f"\nTotal variance: {total_variance:.4f}")
print(f"\nModel converged: {result.converged}")